<a href="https://colab.research.google.com/github/Oorozcoh/Colab_VotaConCiencia/blob/main/EntregaFinal_Generacion_de_Prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI
from openai import OpenAIError

# Inicialización del cliente OpenAI con manejo de errores básico
try:
    # Se recomienda usar variables de entorno o ingresar la API key de forma controlada
    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print("Cliente de OpenAI inicializado correctamente.")
except Exception as e:
    print(f"Error al inicializar el cliente de OpenAI: {e}")

Cliente de OpenAI inicializado correctamente.


In [ ]:
def analizar_propuesta_estudiantil(nombre_candidato, ciclo, texto_propuesta):
    # Prompt estructurado utilizando técnicas de rol y delimitadores
    system_prompt = (
        "Eres un comité evaluador experto en viabilidad de proyectos escolares. "
        "Analiza la propuesta del candidato considerando los recursos disponibles, "
        "el tiempo de ejecución y la viabilidad técnica en un entorno escolar. "
        "Devuelve un análisis constructivo y un puntaje de viabilidad."
    )

    user_prompt = (
        f"Candidato: {nombre_candidato}\n"
        f"Ciclo: {ciclo}\n"
        f"Propuesta: \"{texto_propuesta}\""
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", # Modelo eficiente y económico para este tipo de tareas
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=500
        )

        # Extracción del contenido generado
        resultado_texto = response.choices[0].message.content

        # Medición de tokens (requerimiento clave de eficiencia)
        usage = response.usage
        prompt_tokens = usage.prompt_tokens
        completion_tokens = usage.completion_tokens
        total_tokens = usage.total_tokens

        # Cálculo estimado de costos (ejemplo basado en tarifas estándar de gpt-4o-mini: ~$0.15 / 1M input tokens, ~$0.60 / 1M output tokens)
        costo_estimado = (prompt_tokens * 0.00000015) + (completion_tokens * 0.00000060)

        print(f"--- Análisis para {nombre_candidato} ---")
        print(resultado_texto)
        print("\n[Métricas de Eficiencia de la API]")
        print(f"- Tokens de entrada (Prompt): {prompt_tokens}")
        print(f"- Tokens de salida (Completion): {completion_tokens}")
        print(f"- Tokens totales consumidos: {total_tokens}")
        print(f"- Costo estimado de la llamada: ${costo_estimado:.6f} USD\n")

        return resultado_texto

    except OpenAIError as e:
        print(f"Error de la API de OpenAI: {e}")
        return None
    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")
        return None

In [ ]:
# Casos de prueba reales basados en propuestas del entorno escolar
propuestas_prueba = [
    {
        "nombre": "Sofia Agudelo Correa",
        "ciclo": "1.2",
        "propuesta": "Implementar un sistema de reciclaje de papel automatizado con puntos ecológicos en cada pasillo del colegio y premiar al curso que más recicle al mes."
    },
    {
        "nombre": "Isaac Quinchia Mejia",
        "ciclo": "3.1",
        "propuesta": "Crear un torneo intercolegiado de videojuegos educativos de programación durante los descansos utilizando las tabletas de la institución."
    }
]

# Ejecución de la prueba unitaria
for p in propuestas_prueba:
    analizar_propuesta_estudiantil(p["nombre"], p["ciclo"], p["propuesta"])

--- Análisis para Sofia Agudelo Correa ---
**Análisis de la Propuesta de Sofia Agudelo Correa**

**Recursos Disponibles:**
La implementación de un sistema de reciclaje automatizado requerirá ciertos recursos materiales, como contenedores de reciclaje, dispositivos para la automatización (sensores, sistemas de pesaje, etc.) y posiblemente software para el seguimiento de los datos de reciclaje. Es fundamental evaluar si la institución cuenta con un presupuesto para adquirir estos recursos o si se pueden obtener a través de donaciones o colaboraciones con empresas locales.

**Tiempo de Ejecución:**
La propuesta implica varias etapas: diseño del sistema, adquisición de materiales, instalación de los puntos ecológicos y la planificación de un sistema de premiación. Dependiendo del tamaño del colegio y la complejidad del sistema, el tiempo de ejecución podría variar. Es recomendable establecer un cronograma claro y factible que contemple cada fase del proyecto, así como un periodo de capacit

In [ ]:
try:
    # Generación de imagen utilizando DALL-E 3
    response_imagen = client.images.generate(
        model="dall-e-3",
        prompt="Un póster estudiantil futurista y limpio, estilo corporativo moderno y tecnológico, con acentos en tonos cian y oscuros, que promueva la innovación y la participación democrática en un colegio.",
        size="1024x1024",
        quality="standard",
        n=1,
    )

    image_url = response_imagen.data[0].url
    print("¡Imagen generada con éxito!")
    print("URL de acceso temporal:", image_url)

except Exception as e:
    print(f"Error al generar la imagen con OpenAI: {e}")

Error al generar la imagen con OpenAI: Error code: 400 - {'error': {'message': "The model 'dall-e-3' does not exist.", 'type': 'image_generation_user_error', 'param': 'model', 'code': 'invalid_value'}}


In [ ]:
# ==========================================
# MÓDULO DE GENERACIÓN VISUAL (TEXTO A IMAGEN)
# ==========================================

prompt_imagen = (
    "Un póster estudiantil futurista y limpio, estilo corporativo moderno y tecnológico, "
    "con acentos en tonos cian y oscuros, que promueva la innovación y la participación "
    "democrática en un colegio."
)

print(f"Prompt estructurado para la generación visual:\n-> '{prompt_imagen}'")

In [ ]:
!pip install huggingface_hub

In [ ]:
import requests
from io import BytesIO
from PIL import Image
from IPython.display import display

def generar_afiche_estudiantil(prompt_usuario):
    """
    Genera una imagen de campaña escolar utilizando una API pública de inferencia gratuita,
    mostrando el resultado en el Notebook.
    """
    prompt_formateado = prompt_usuario.replace(" ", "%20")
    url = f"https://image.pollinations.ai/prompt/{prompt_formateado}?width=512&height=512&nologo=true"

    response = requests.get(url)

    if response.status_code == 200:
        image = Image.open(BytesIO(response.content))
        return image
    else:
        raise Exception(f"Error al generar la imagen: {response.status_code}")

# Ejemplo de uso en tu Notebook para la POC de "VotaConCiencia"
prompt_propuesta = "A vibrant school poster about modernizing the school cafeteria, clean vector art style, friendly colors, educational campaign"

resultado = generar_afiche_estudiantil(prompt_propuesta)
display(resultado)